# Working with Datasets

The `medical_image` framework provides PyTorch-compatible datasets for medical image collections. This notebook shows how to use the INbreast and Custom INbreast datasets.

In [ ]:
!pip install medical-image-std

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from medical_image.datasets.inbreast import INbreastDataset
from medical_image.datasets.custom_inbreast import CustomINbreastDataset

## 1. INbreast Dataset

The INbreast dataset expects the following directory structure:

```
INbreast Release 1.0/
├── AllDICOMs/
│   ├── 20586908_*.dcm
│   └── ...
├── AllXML/
│   ├── 20586908.xml
│   └── ...
├── AllROI/       (optional)
└── INbreast.csv
```

In [ ]:
# Load the INbreast dataset
# Replace with your actual path
INBREAST_ROOT = "path/to/INbreast Release 1.0"

dataset = INbreastDataset(
    root_dir=INBREAST_ROOT,
    target_size=(512, 512),  # resize all images to 512x512
)

print(f"Dataset size: {len(dataset)} samples")

In [ ]:
# Access a single sample
sample = dataset[0]

image = sample["image"]  # shape: (1, H, W)
mask = sample["mask"]  # shape: (1, H, W)
meta = sample["metadata"]

print(f"Image shape: {image.shape}")
print(f"Mask shape:  {mask.shape}")
print(f"Metadata:    {meta}")

In [ ]:
# Visualize image and mask
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image.squeeze(0).numpy(), cmap="gray")
axes[0].set_title(f"Image ({meta['case_id']})")
axes[1].imshow(mask.squeeze(0).numpy(), cmap="gray")
axes[1].set_title("Segmentation Mask")
plt.tight_layout()
plt.show()

## 2. Using a DataLoader for Training

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

batch = next(iter(loader))
print(f"Batch image shape: {batch['image'].shape}")  # (4, 1, 512, 512)
print(f"Batch mask shape:  {batch['mask'].shape}")  # (4, 1, 512, 512)

## 3. Custom INbreast Dataset (with TIF Masks)

Expected structure:

```
custom_root/
├── AllMasks/
│   ├── 20586934_mask.tif
│   └── ...
└── INbreast Release 1.0/
    ├── AllDICOMs/
    ├── AllXML/
    └── AllROI/
```

In [ ]:
CUSTOM_ROOT = "path/to/custom_inbreast"

custom_ds = CustomINbreastDataset(
    root_dir=CUSTOM_ROOT,
    target_size=(512, 512),
)

sample = custom_ds[0]
print(f"Mask source: {sample['metadata']['mask_source']}")  # 'tif', 'xml', or 'empty'

## 4. Applying Transforms

In [ ]:
import torchvision.transforms as T

# Define image-level transforms
image_transform = T.Compose(
    [
        T.Normalize(mean=[0.5], std=[0.5]),
    ]
)

dataset_aug = INbreastDataset(
    root_dir=INBREAST_ROOT,
    target_size=(256, 256),
    transform=image_transform,
)

sample = dataset_aug[0]
print(
    f"Normalized pixel range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]"
)

## 5. COCO JSON Export/Import

Datasets can export annotations to COCO format for interoperability.

In [ ]:
# Export to COCO JSON
coco_dict = dataset.to_coco_json(
    output_path="inbreast_coco.json",
    description="INbreast mammography dataset",
)
print(f"Images: {len(coco_dict['images'])}")
print(f"Annotations: {len(coco_dict['annotations'])}")
print(f"Categories: {len(coco_dict['categories'])}")

In [ ]:
# Import from COCO JSON
from medical_image.datasets.base_dataset import BaseDataset

coco_data = BaseDataset.from_coco_json("inbreast_coco.json")
print(f"Loaded {len(coco_data['images'])} images")
print(f"Categories: {coco_data['categories']}")

## 6. Downloading a Dataset

In [ ]:
# Download from a local path or URL
# dest = INbreastDataset.download(
#     source="/shared/datasets/INbreast.tar.gz",
#     destination="./data/inbreast",
#     method="local",
# )
# print(f"Downloaded to: {dest}")